In [ ]:
from astropy.table import Table
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pymc3 as pm
import pymc3_ext as pmx
import exoplanet as xo
import os
from corner import corner
import theano.tensor as tt
from tqdm.notebook import tqdm
from scipy.stats import median_abs_deviation
import pandas as pd
from astroquery.mast import Catalogs
from scipy.interpolate import UnivariateSpline
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
path_cat = "/home/luciam/Documents/starry/Resultados_Orion/catalogo_completo_mist_sommers.csv"

cat = pd.read_csv(path_cat)
cat.head

In [ ]:

# ============================================================
# 1. Cargar priors fotométricos
# ============================================================

path_priors = "/home/luciam/Documents/starry/Orion_yceb/priors.csv"

priors = pd.read_csv(path_priors)

# ============================================================
# 2. TIC seleccionado
# ============================================================

tic_id = "TIC 138829403"
tic_num = int(tic_id.replace("TIC ", ""))

# Buscar el TIC
row = priors.loc[priors["TIC"] == tic_num]

if len(row) == 0:
    raise ValueError(f"{tic_id} no fue encontrado en priors.csv")

prior = row.iloc[0]

# ============================================================
# 3. Extraer parámetros fotométricos
# ============================================================

P_orb = float(prior["P_orb"])

depth_primary = float(prior["Depth_primary"])
depth_primary_err = float(prior["Depth_primary_err"])

depth_secondary = float(prior["Depth_secondary"])
depth_secondary_err = float(prior["Depth_secondary_err"])

duration = float(prior["Duration"])
t0 = float(prior["t0"])

# ============================================================
# 4. Mostrar resultados
# ============================================================

print("=" * 65)
print(f"PRIORS FOTOMÉTRICOS — {tic_id}")
print("=" * 65)

print(f"Periodo orbital         = {P_orb:.6f} d")
print(f"t0                      = {t0:.6f}")
print(f"Duración eclipse        = {duration:.6f} d")

print()
print(f"Profundidad primaria    = "
      f"{depth_primary:.6f} ± {depth_primary_err:.6f}")

print(f"Profundidad secundaria  = "
      f"{depth_secondary:.6f} ± {depth_secondary_err:.6f}")

print("=" * 65)

# ============================================================
# 5. Cargar catálogo MIST + Somers
# ============================================================

path_catalog = (
    "/home/luciam/Documents/starry/"
    "Resultados_Orion/catalogo_completo_mist_sommers.csv"
)

cat = pd.read_csv(path_catalog)

# Asegurar que TIC sea numérico
cat["TIC"] = pd.to_numeric(cat["TIC"], errors="coerce")

# Buscar el TIC seleccionado
star_row = cat.loc[cat["TIC"] == tic_num]

if len(star_row) == 0:
    raise ValueError(
        f"{tic_id} no fue encontrado en "
        "catalogo_completo_mist_sommers"
    )

star = star_row.iloc[0]


# ============================================================
# 6. Constante solar
# ============================================================

T_SUN = 5772.0  # K


# ============================================================
# 7. Parámetros MIST de la componente dominante
# ============================================================

T_dom_MIST = float(star["T_median_1"])
sigma_T_dom_MIST = float(star["T_std_1"])

M_dom_MIST = float(star["mass_median_1"])
sigma_M_dom_MIST = float(star["mass_std_1"])

age_dom_MIST = float(star["age_median_1"])
sigma_age_dom_MIST = float(star["age_std_1"])

logL_dom_MIST = float(star["lgL_median_1"])
sigma_logL_dom_MIST = float(star["lgL_std_1"])


# ============================================================
# 8. Radio de la componente dominante — MIST
# ============================================================

# Luminosidad en unidades solares
L_dom_MIST = 10**logL_dom_MIST

# Radio en unidades solares
R_dom_MIST = (
    np.sqrt(L_dom_MIST)
    * (T_dom_MIST / T_SUN)**(-2)
)


# ============================================================
# 9. Incertidumbre del radio — MIST
#
# R ∝ L^(1/2) T^(-2)
#
# Como tenemos log10(L):
#
# sigma_ln(L) = ln(10) * sigma_logL
#
# ============================================================

frac_sigma_R_MIST = np.sqrt(
    (
        0.5 * np.log(10) * sigma_logL_dom_MIST
    )**2
    +
    (
        2.0 * sigma_T_dom_MIST / T_dom_MIST
    )**2
)

sigma_R_dom_MIST = R_dom_MIST * frac_sigma_R_MIST


# ============================================================
# 10. Parámetros Somers 2017 + 17% spots
# ============================================================

T_dom_SOMERS = float(star["T_median_2"])
sigma_T_dom_SOMERS = float(star["T_std_2"])

M_dom_SOMERS = float(star["mass_median_2"])
sigma_M_dom_SOMERS = float(star["mass_std_2"])

age_dom_SOMERS = float(star["age_median_2"])
sigma_age_dom_SOMERS = float(star["age_std_2"])

logL_dom_SOMERS = float(star["lgL_median_2"])
sigma_logL_dom_SOMERS = float(star["lgL_std_2"])


# ============================================================
# 11. Radio de la componente dominante — Somers
# ============================================================

L_dom_SOMERS = 10**logL_dom_SOMERS

R_dom_SOMERS = (
    np.sqrt(L_dom_SOMERS)
    * (T_dom_SOMERS / T_SUN)**(-2)
)


# ============================================================
# 12. Incertidumbre del radio — Somers
# ============================================================

frac_sigma_R_SOMERS = np.sqrt(
    (
        0.5 * np.log(10) * sigma_logL_dom_SOMERS
    )**2
    +
    (
        2.0 * sigma_T_dom_SOMERS / T_dom_SOMERS
    )**2
)

sigma_R_dom_SOMERS = R_dom_SOMERS * frac_sigma_R_SOMERS


# ============================================================
# 13. Mostrar resultados
# ============================================================

print("\n" + "=" * 70)
print(f"PROPIEDADES DE LA COMPONENTE DOMINANTE — {tic_id}")
print("=" * 70)

print("\nMIST")
print("-" * 70)
print(
    f"Teff       = {T_dom_MIST:.2f} ± "
    f"{sigma_T_dom_MIST:.2f} K"
)
print(
    f"Masa       = {M_dom_MIST:.4f} ± "
    f"{sigma_M_dom_MIST:.4f} Msun"
)
print(
    f"Edad       = {age_dom_MIST:.4f} ± "
    f"{sigma_age_dom_MIST:.4f} Myr"
)
print(
    f"log(L/Lsun)= {logL_dom_MIST:.4f} ± "
    f"{sigma_logL_dom_MIST:.4f}"
)
print(
    f"Radio      = {R_dom_MIST:.4f} ± "
    f"{sigma_R_dom_MIST:.4f} Rsun"
)

print("\nSomers 2017 + 17% spots")
print("-" * 70)
print(
    f"Teff       = {T_dom_SOMERS:.2f} ± "
    f"{sigma_T_dom_SOMERS:.2f} K"
)
print(
    f"Masa       = {M_dom_SOMERS:.4f} ± "
    f"{sigma_M_dom_SOMERS:.4f} Msun"
)
print(
    f"Edad       = {age_dom_SOMERS:.4f} ± "
    f"{sigma_age_dom_SOMERS:.4f} Myr"
)
print(
    f"log(L/Lsun)= {logL_dom_SOMERS:.4f} ± "
    f"{sigma_logL_dom_SOMERS:.4f}"
)
print(
    f"Radio      = {R_dom_SOMERS:.4f} ± "
    f"{sigma_R_dom_SOMERS:.4f} Rsun"
)

print("=" * 70)

In [ ]:

path_modelos = Path(
    "/home/luciam/Documents/starry/"
    "Resultados_Orion/datos_estrellas_massage_orion/modelos"
)

# ============================================================
# MIST
# ============================================================

mist = pd.read_csv(path_modelos / "MIST_V2.csv")

print("=" * 80)
print("MIST_V2.csv")
print("=" * 80)
print(f"Dimensiones: {mist.shape}")
print("\nColumnas:")
print(mist.columns.tolist())

print("\nPrimeras filas:")
display(mist.head())


# ============================================================
# Somers / SPOTS f = 17%
# ============================================================

somers = pd.read_csv(path_modelos / "f017track_ms.csv")

print("\n" + "=" * 80)
print("f017track_ms.csv")
print("=" * 80)
print(f"Dimensiones: {somers.shape}")
print("\nColumnas:")
print(somers.columns.tolist())

print("\nPrimeras filas:")
display(somers.head())

In [ ]:
path_modelos = Path(
    "/home/luciam/Documents/starry/"
    "Resultados_Orion/datos_estrellas_massage_orion/modelos"
)

mist = pd.read_csv(path_modelos / "MIST_V2.csv")
somers = pd.read_csv(path_modelos / "f017track_ms.csv")

# ============================================================
# MIST
# ============================================================

print("=" * 80)
print("MIST V2")
print("=" * 80)

print("Edad:")
print(f"  min = {mist['star_age'].min():.4e}")
print(f"  max = {mist['star_age'].max():.4e}")

print("\nMasa:")
print(f"  min = {mist['star_mass'].min():.4f}")
print(f"  max = {mist['star_mass'].max():.4f}")

print("\nlog L:")
print(f"  min = {mist['log_L'].min():.4f}")
print(f"  max = {mist['log_L'].max():.4f}")

print("\nlog Teff:")
print(f"  min = {mist['log_Teff'].min():.4f}")
print(f"  max = {mist['log_Teff'].max():.4f}")

print("\nFases:")
print(mist["phase"].value_counts().sort_index())


# ============================================================
# SOMERS
# ============================================================

print("\n" + "=" * 80)
print("SOMERS / SPOTS fspot = 17%")
print("=" * 80)

print("Edad [Gyr]:")
print(f"  min = {somers['Age(Gyr)'].min():.6f}")
print(f"  max = {somers['Age(Gyr)'].max():.6f}")

print("\nMasa:")
print(f"  min = {somers['Mass'].min():.4f}")
print(f"  max = {somers['Mass'].max():.4f}")

print("\nFspot:")
print(somers["Fspot"].unique())

print("\nNúmero de masas:")
print(somers["Mass"].nunique())

print("\nPrimeras masas:")
print(np.sort(somers["Mass"].unique())[:20])

print("\nÚltimas masas:")
print(np.sort(somers["Mass"].unique())[-20:])

In [ ]:
# ============================================================
# INSPECCIÓN DE UNA TRACK MIST
# ============================================================

mass_test = 0.5

track = mist.loc[
    np.isclose(mist["star_mass"], mass_test, atol=1e-3)
].copy()

track = track.sort_values("star_age")

print(f"Número de puntos para M = {mass_test} Msun: {len(track)}")

display(track.head(10))
display(track.tail(10))

In [ ]:
# ============================================================
# RANGO EVOLUTIVO DE LA TRACK
# ============================================================

print("Edad [Myr]:")
print(
    track["star_age"].min() / 1e6,
    "→",
    track["star_age"].max() / 1e6
)

print("\nTeff [K]:")
print(
    10**track["log_Teff"].min(),
    "→",
    10**track["log_Teff"].max()
)

print("\nRadio [Rsun]:")
print(
    10**track["log_R"].min(),
    "→",
    10**track["log_R"].max()
)

print("\nLuminosidad [Lsun]:")
print(
    10**track["log_L"].min(),
    "→",
    10**track["log_L"].max()
)

print("\nFases:")
print(track["phase"].value_counts().sort_index())

In [ ]:
mist_masses = np.sort(mist["star_mass"].unique())

print(f"Número de masas únicas en MIST: {len(mist_masses)}")

print("\nPrimeras masas:")
print(mist_masses[:30])

print("\nÚltimas masas:")
print(mist_masses[-30:])

In [ ]:
# Track ~0.5 Msun
mass_test = 0.5

track = mist.loc[
    np.isclose(mist["star_mass"], mass_test, atol=1e-3)
].copy()

track = track.sort_values("star_age")

# Convertir unidades
track["Age_Myr"] = track["star_age"] / 1e6
track["Teff_K"] = 10**track["log_Teff"]
track["L_Lsun"] = 10**track["log_L"]
track["R_Rsun"] = 10**track["log_R"]

fig, ax = plt.subplots(figsize=(8, 6))

for phase, group in track.groupby("phase"):
    ax.plot(
        group["Teff_K"],
        group["L_Lsun"],
        marker="o",
        markersize=2,
        linewidth=1,
        label=f"phase = {phase:g}"
    )

ax.set_yscale("log")
ax.set_xscale("log")

# En HR diagram, estrellas más calientes a la izquierda
ax.invert_xaxis()

ax.set_xlabel(r"$T_{\rm eff}$ [K]", fontsize=13)
ax.set_ylabel(r"$L/L_\odot$", fontsize=13)

ax.set_title(r"MIST track — $\sim0.5\,M_\odot$", fontsize=14)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# MIST: distribución de puntos por fase
# ============================================================

for phase in np.sort(mist["phase"].unique()):

    sub = mist[mist["phase"] == phase]

    print("=" * 70)
    print(f"PHASE = {phase}")
    print("=" * 70)

    print(f"N puntos       : {len(sub):,}")
    print(f"Masa min       : {sub['star_mass'].min():.4f}")
    print(f"Masa max       : {sub['star_mass'].max():.4f}")

    print(
        f"Edad min [Myr] : "
        f"{sub['star_age'].min()/1e6:.6f}"
    )

    print(
        f"Edad max [Myr] : "
        f"{sub['star_age'].max()/1e6:.3f}"
    )

    print()

In [ ]:
# ============================================================
# INSPECCIÓN DE PHASE PARA ESTRELLAS DE BAJA MASA
# ============================================================

for mass_test in [0.2, 0.5, 0.8, 1.0, 1.2]:

    sub = mist.loc[
        np.isclose(mist["star_mass"], mass_test, atol=0.002)
    ].copy()

    if len(sub) == 0:
        print(f"No hay puntos para ~{mass_test} Msun")
        continue

    sub["Age_Myr"] = sub["star_age"] / 1e6
    sub["Teff_K"] = 10**sub["log_Teff"]
    sub["L_Lsun"] = 10**sub["log_L"]
    sub["R_Rsun"] = 10**sub["log_R"]

    print("\n" + "=" * 70)
    print(f"M ~ {mass_test} Msun")
    print("=" * 70)

    for phase in sorted(sub["phase"].unique()):

        p = sub[sub["phase"] == phase]

        print(
            f"phase={phase:4.1f} | "
            f"N={len(p):4d} | "
            f"Age={p['Age_Myr'].min():.5f}"
            f"–{p['Age_Myr'].max():.3f} Myr | "
            f"T={p['Teff_K'].min():.0f}"
            f"–{p['Teff_K'].max():.0f} K | "
            f"R={p['R_Rsun'].min():.3f}"
            f"–{p['R_Rsun'].max():.3f} Rsun"
        )

In [ ]:
# ============================================================
# MASAS APROXIMADAS DE LAS TRACKS MIST
# ============================================================

mist["mass_grid_approx"] = mist["star_mass"].round(2)

mass_summary = (
    mist.groupby("mass_grid_approx")
    .agg(
        n=("star_age", "size"),
        age_min_Myr=("star_age", lambda x: x.min()/1e6),
        age_max_Myr=("star_age", lambda x: x.max()/1e6),
        mass_min=("star_mass", "min"),
        mass_max=("star_mass", "max"),
    )
    .reset_index()
)

display(
    mass_summary[
        (mass_summary["mass_grid_approx"] >= 0.1) &
        (mass_summary["mass_grid_approx"] <= 1.3)
    ].head(30)
)

In [ ]:
print(mist.columns.tolist())

In [ ]:
# ============================================================
# PREPARAR VARIABLES FÍSICAS
# ============================================================

mist["Age_Myr"] = mist["star_age"] / 1e6
mist["R_Rsun"] = 10**mist["log_R"]
mist["Teff_K"] = 10**mist["log_Teff"]

# ============================================================
# TRACKS PMS
# ============================================================

mist_pms = mist[mist["phase"] == -1].copy()

print("Modelos PMS:", len(mist_pms))
print("Edad:", mist_pms["Age_Myr"].min(), "-", mist_pms["Age_Myr"].max(), "Myr")
print("Radio:", mist_pms["R_Rsun"].min(), "-", mist_pms["R_Rsun"].max(), "Rsun")
print("Teff:", mist_pms["Teff_K"].min(), "-", mist_pms["Teff_K"].max(), "K")

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    mist_pms["R_Rsun"],
    mist_pms["Teff_K"],
    s=1,
    alpha=0.15
)

plt.xscale("log")
plt.xlabel(r"$R\ [R_\odot]$")
plt.ylabel(r"$T_{\rm eff}\ [K]$")
plt.title("MIST PMS")

plt.show()

In [ ]:
# ============================================================
# DOMINIO JOVEN PARA LA ISÓCRONA / INTERPOLADOR
# ============================================================

mist_young = mist[
    (mist["mass_grid_approx"] >= 0.10) &
    (mist["mass_grid_approx"] <= 1.30) &
    (mist["Age_Myr"] <= 100)
].copy()

print("Modelos:", len(mist_young))

print(
    f"Edad: {mist_young['Age_Myr'].min():.4f} - "
    f"{mist_young['Age_Myr'].max():.2f} Myr"
)

print(
    f"Radio: {mist_young['R_Rsun'].min():.3f} - "
    f"{mist_young['R_Rsun'].max():.2f} Rsun"
)

print(
    f"Teff: {mist_young['Teff_K'].min():.0f} - "
    f"{mist_young['Teff_K'].max():.0f} K"
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

sc = plt.scatter(
    mist_young["R_Rsun"],
    mist_young["Teff_K"],
    c=mist_young["Age_Myr"],
    s=2,
    alpha=0.4
)

plt.xscale("log")
plt.yscale("log")

plt.xlabel(r"$R\ [R_\odot]$")
plt.ylabel(r"$T_{\rm eff}\ [K]$")
plt.title("MIST PMS: $0.1-1.3\,M_\\odot$, $t\\leq100$ Myr")

cbar = plt.colorbar(sc)
cbar.set_label("Age [Myr]")

plt.show()

In [ ]:
age_test = 20

# Tomamos modelos alrededor de 20 Myr
tol = 0.5

test = mist_young[
    np.abs(mist_young["Age_Myr"] - age_test) < tol
].copy()

print("Modelos:", len(test))

plt.figure(figsize=(8, 6))

plt.scatter(
    test["R_Rsun"],
    test["Teff_K"],
    s=8
)

plt.xscale("log")
plt.yscale("log")

plt.xlabel(r"$R\ [R_\odot]$")
plt.ylabel(r"$T_{\rm eff}\ [K]$")
plt.title(r"MIST PMS alrededor de $t=20$ Myr")

plt.show()

In [ ]:
# ¿A qué edades pertenece la banda rara?
raros = mist_young[mist_young["Teff_K"] > 15000]
print(raros["Age_Myr"].describe())
print(raros["mass_grid_approx"].describe())
print(raros["star_mass"].describe())  # comparar con mass_grid_approx

In [ ]:
raros_muestra = mist_young[mist_young["Teff_K"] > 15000].sample(10, random_state=0)
print(raros_muestra[["star_mass", "mass_grid_approx", "Age_Myr",
                      "star_age", "log_R", "R_Rsun",
                      "log_Teff", "Teff_K", "phase"]])

In [ ]:
print(mist.index.duplicated().sum())
print(mist.index.is_unique)


path_modelos = Path(
    "/home/luciam/Documents/starry/Resultados_Orion/datos_estrellas_massage_orion/modelos"
)
mist = pd.read_csv(path_modelos / "MIST_V2.csv")


In [ ]:
print(mist.columns.tolist())

In [ ]:
# Reemplaza con la ruta/nombre real de tu archivo EEP para masa ~1.08 M_sun
import pandas as pd


raw["Age_Myr"] = raw["star_age"] / 1e6

# Busca la fila con edad más cercana a 61.35 Myr
fila = raw.iloc[(raw["Age_Myr"] - 61.35).abs().argmin()]
print(fila[["star_age", "star_mass", "log_R", "log_Teff", "phase"]])

In [ ]:
tracks = [pd.read_csv(f) for f in archivos_eep]  # cada uno con su propio índice 0..N
mist = pd.concat(tracks)          # sin ignore_index=True       # sin ignore_index=True